In [2]:
# Name : CHANDRAPRIYADHARSHINI C
# Registor. no : 212223240019
import warnings ; warnings.filterwarnings('ignore')

import gym
import numpy as np

import random
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
np.set_printoptions(suppress=True)
random.seed(123); np.random.seed(123);

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
def print_policy(pi, P, action_symbols=('<', 'v', '>', '^'), n_cols=4, title='Policy:'):
    print(title)
    arrs = {k:v for k,v in enumerate(action_symbols)}
    for s in range(len(P)):
        a = pi(s)
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), arrs[a].rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")


In [4]:
def print_state_value_function(V, P, n_cols=4, prec=3, title='State-value function:'):
    print(title)
    for s in range(len(P)):
        v = V[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), '{}'.format(np.round(v, prec)).rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [5]:
def probability_success(env, pi, goal_state, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        while not done and steps < max_steps:
            state, _, done, h = env.step(pi(state))
            steps += 1
        results.append(state == goal_state)
    return np.sum(results)/len(results)

In [6]:
def mean_return(env, pi, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        results.append(0.0)
        while not done and steps < max_steps:
            state, reward, done, _ = env.step(pi(state))
            results[-1] += reward
            steps += 1
    return np.mean(results)

In [7]:
env = gym.make('FrozenLake-v1')
P = env.env.P
init_state = env.reset()
goal_state = 15
LEFT, DOWN, RIGHT, UP = range(4)

In [8]:
P

{0: {0: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  2: [(0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)],
  3: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 1: {0: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False)],
  2: [(0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  3: [(0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 2:

In [9]:
init_state

0

In [10]:
pi_frozenlake = lambda s: {
    0: RIGHT,
    1: DOWN,
    2: RIGHT,
    3: LEFT,
    4: DOWN,
    5: LEFT,
    6: RIGHT,
    7:LEFT,
    8: UP,
    9: DOWN,
    10:LEFT,
    11:DOWN,
    12:RIGHT,
    13:RIGHT,
    14:DOWN,
    15:LEFT #Stop
}[s]
print_policy(pi_frozenlake, P, action_symbols=('<', 'v', '>', '^'), n_cols=4)

Policy:
| 00      > | 01      v | 02      > | 03      < |
| 04      v |           | 06      > |           |
| 08      ^ | 09      v | 10      < |           |
|           | 13      > | 14      v |           |


In [12]:
print('Reaches goal {:.2f}%. Obtains an average undiscounted return of {:.4f}.'.format(probability_success(env, pi_frozenlake, goal_state=goal_state) * 100,mean_return(env, pi_frozenlake)))

Reaches goal 10.00%. Obtains an average undiscounted return of 0.1000.


In [13]:
def policy_evaluation(pi, P, gamma=1.0, theta=1e-10):
    V = np.zeros(len(P), dtype=np.float64)

    while True:
        delta = 0
        for s in range(len(P)):
            v = 0
            a = pi(s)
            for prob, next_state, reward, done in P[s][a]:
                v += prob * (reward + gamma * V[next_state])
            delta = max(delta, abs(V[s] - v))
            V[s] = v

        if delta < theta:
            break

    return V


In [14]:
V1 = policy_evaluation(pi_frozenlake, P,gamma=0.99)
print_state_value_function(V1, P, n_cols=4, prec=5)

State-value function:
| 00 0.11448 | 01 0.08191 | 02 0.13372 | 03 0.06586 |
| 04 0.15053 |           | 06 0.20562 |           |
| 08 0.30562 | 09 0.46997 | 10 0.48938 |           |
|           | 13 0.62915 | 14 0.80739 |           |


In [15]:
from tqdm import tqdm

In [16]:
def decay_schedule(init_value, min_value, decay_ratio, max_steps, log_start=-2,log_base=10):
  decay_steps=int(max_steps * decay_ratio)
  rem_steps=max_steps-decay_steps

  values=np.logspace(log_start,0,decay_steps,base=log_base,endpoint=True)[::-1]
  values=(values-values.min())/(values.max()-values.min())
  values=(init_value-min_value)*values+min_value
  values=np.pad(values,(0,rem_steps),'edge')
  return values

In [17]:
from itertools import count
def generate_trajectory (pi, env, max_steps=20):
  done, trajectory = False, []
  while not done:
    state = env.reset()
    for t in count():
      action = pi (state)
      next_state, reward, done,_=env.step(action)
      experience=(state, action, reward,next_state,done)
      trajectory.append(experience)
      if done:
        break
      if t>=max_steps-1:
        trajectory=[]
        break
      state=next_state
  return np.array(trajectory,object)

In [18]:
def mc_prediction(pi,env,gamma=1.0,init_alpha=0.5,min_alpha=0.01,
                  alpha_decay_ratio=0.3,
                  n_episodes=50000,max_steps=100,first_visit=True):
  nS=env.observation_space.n
  discounts=np.logspace(0,max_steps,num=max_steps,base=gamma,endpoint=False)
  alphas=decay_schedule(init_alpha,min_alpha,alpha_decay_ratio,n_episodes)
  V=np.zeros(nS)
  V_track=np.zeros((n_episodes,nS))
  for e in tqdm(range(n_episodes),leave=False):
    trajectory=generate_trajectory(pi,env,max_steps)
    visited=np.zeros(nS,dtype=np.bool)
    for t, (state, _, reward, _, _) in enumerate(trajectory):
      if visited[state] and first_visit:
        continue
      visited[state]=True
      n_steps=len(trajectory[t:])
      G=np.sum(discounts[:n_steps]*trajectory[t:,2])
      V[state]=V[state]+alphas[e]*(G-V[state])
    V_track[e]=V
  return V.copy(),V_track

In [19]:
v_mc, v2_mc = mc_prediction(pi_frozenlake, env)

In [30]:
print("Name :  CHANDRAPRIYADHARSHINI C")
print("Reg.No: 212223240019")
print()
print_state_value_function(v_mc, P, n_cols=4, prec=5)

Name :  CHANDRAPRIYADHARSHINI C
Reg.No: 212223240019

State-value function:
| 00 0.14934 | 01 0.11804 | 02 0.1946 | 03 0.10651 |
| 04 0.16515 |           | 06 0.27645 |           |
| 08 0.32961 | 09 0.49881 | 10 0.56841 |           |
|           | 13 0.67456 | 14 0.83559 |           |


In [21]:
print(v2_mc)

[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.15236783 0.11803956 0.19460183 ... 0.67456093 0.83559407 0.        ]
 [0.15084415 0.11803956 0.19460183 ... 0.67456093 0.83559407 0.        ]
 [0.14933571 0.11803956 0.19460183 ... 0.67456093 0.83559407 0.        ]]


In [29]:
print("Name :  CHANDRAPRIYADHARSHINI C")
print("Reg.No: 212223240019")
print()
if np.sum(v_mc > V1) == 11:
    print("\nThe first policy is the better policy.")
elif np.sum(V1 > v_mc) == 11:
    print("\nYour policy is the better policy.")
else:
    print("\nBoth policies have their merits.")

Name :  CHANDRAPRIYADHARSHINI C
Reg.No: 212223240019


The first policy is the better policy.
